# Encoder–Decoder Transformer: Complete Pipeline Lab

## Full Pipeline

### Training
`Input Text Corpus → Tokenization → Embedding → Positional Encoding → Encoder Stack → Encoder Output → Decoder Stack (Masked Self-Attention + Cross-Attention + FFN) → Output Token Prediction Head → Loss → Backpropagation → Saved Weights`

### Inference
`User Input → Tokenization → Embedding → Positional Encoding → Encoder Stack → Encoder Output → Decoder Stack → Token-by-Token Generation`

---

We will build each stage one by one.

---
## Step 1: Input Text Corpus

### What?
The raw parallel text data that the model learns from. For encoder-decoder models, we need **source-target pairs** (e.g., English → French for translation, or document → summary for summarization).

### Why?
- The encoder-decoder architecture is designed for **sequence-to-sequence** tasks — the encoder reads the source, the decoder generates the target.
- Without paired data, the model has no signal to learn the mapping from input to output.

### Where in the pipeline?
```
👉 [Input Text Corpus] → Tokenization → Embedding → Positional Encoding → Encoder → Decoder → Output
```

### Real-world examples
| Task | Source (Encoder Input) | Target (Decoder Input/Output) |
|------|----------------------|------------------------------|
| Translation | "How are you?" | "Comment allez-vous?" |
| Summarization | Long article text | Short summary |
| Q&A | Question + Context | Answer |
| Text-to-SQL | "Show all users" | "SELECT * FROM users" |

In [1]:
# Step 1: Input Text Corpus — Source-Target Pairs

# For encoder-decoder, we always need PAIRS: (source, target)
# The encoder processes the source, the decoder learns to generate the target.

corpus = [
    # (source_text, target_text)
    ("How are you?", "Comment allez-vous?"),
    ("I am fine.", "Je vais bien."),
    ("Thank you very much.", "Merci beaucoup."),
    ("Good morning.", "Bonjour."),
    ("See you tomorrow.", "À demain."),
    ("What is your name?", "Comment vous appelez-vous?"),
    ("I love programming.", "J'adore la programmation."),
    ("The weather is nice today.", "Le temps est beau aujourd'hui."),
]

print(f"Corpus size: {len(corpus)} pairs")
print(f"\nExample pair:")
print(f"  Source (Encoder input):  '{corpus[0][0]}'")
print(f"  Target (Decoder output): '{corpus[0][1]}'")

Corpus size: 8 pairs

Example pair:
  Source (Encoder input):  'How are you?'
  Target (Decoder output): 'Comment allez-vous?'


### Key Takeaway

- Encoder-decoder models need **paired data** (source → target)
- The **encoder** reads the source sequence and builds a rich representation
- The **decoder** uses that representation to generate the target sequence token by token
- Data quality and alignment directly impact model performance

**Next Step →** Tokenization: converting these raw strings into token IDs the model can process.

---
## Step 2: Tokenization

### What?
Converting raw text strings into sequences of **token IDs** (integers) that the model can process. Each unique sub-word/word gets a numeric ID from a vocabulary.

### Why?
- Neural networks only understand numbers, not strings.
- Sub-word tokenization (like BPE) handles **unknown words** gracefully by breaking them into known pieces.
- For encoder-decoder, we tokenize **both** source and target sequences separately.

### Where in the pipeline?
```
Input Text Corpus → 👉 [Tokenization] → Embedding → Positional Encoding → Encoder → Decoder → Output
```

### Special Tokens for Encoder-Decoder
| Token | Purpose |
|-------|---------|
| `<pad>` | Padding shorter sequences to equal length |
| `<sos>` | Start-of-sequence — tells decoder to begin generating |
| `<eos>` | End-of-sequence — tells decoder to stop generating |
| `<unk>` | Unknown token — fallback for unseen words |

### How tokenization differs per component
- **Encoder input**: `[token1, token2, ..., <eos>]` — source sentence
- **Decoder input** (teacher forcing): `[<sos>, target_token1, target_token2, ...]` — shifted right
- **Decoder target** (labels): `[target_token1, target_token2, ..., <eos>]` — what decoder should predict

In [2]:
# Step 2: Tokenization — Building Vocabulary & Converting Text to IDs

import torch
from collections import Counter

# Special tokens
PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN = '<pad>', '<sos>', '<eos>', '<unk>'
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

class Tokenizer:
    """Simple word-level tokenizer for demonstration.
    Production models use BPE/SentencePiece — the concept is identical."""
    
    def __init__(self):
        self.word2idx = {PAD_TOKEN: PAD_IDX, SOS_TOKEN: SOS_IDX, EOS_TOKEN: EOS_IDX, UNK_TOKEN: UNK_IDX}
        self.idx2word = {v: k for k, v in self.word2idx.items()}
    
    def build_vocab(self, sentences):
        """Build vocabulary from list of sentences."""
        for sent in sentences:
            for word in sent.lower().split():
                if word not in self.word2idx:
                    idx = len(self.word2idx)
                    self.word2idx[word] = idx
                    self.idx2word[idx] = word
        print(f"Vocab size: {len(self.word2idx)}")
    
    def encode(self, sentence):
        """Convert sentence string → list of token IDs."""
        return [self.word2idx.get(w, UNK_IDX) for w in sentence.lower().split()]
    
    def decode(self, ids):
        """Convert list of token IDs → sentence string."""
        return ' '.join(self.idx2word.get(i, UNK_TOKEN) for i in ids)

# Build separate vocabs for source (English) and target (French)
src_tokenizer = Tokenizer()
tgt_tokenizer = Tokenizer()

src_tokenizer.build_vocab([s for s, t in corpus])
tgt_tokenizer.build_vocab([t for s, t in corpus])

# Tokenize one example
src_text, tgt_text = corpus[0]
src_ids = src_tokenizer.encode(src_text)
tgt_ids = tgt_tokenizer.encode(tgt_text)

print(f"\nSource: '{src_text}'")
print(f"Source IDs: {src_ids}")
print(f"\nTarget: '{tgt_text}'")
print(f"Target IDs: {tgt_ids}")

# Encoder input:  [token_ids..., EOS]
# Decoder input:  [SOS, token_ids...]       ← shifted right (teacher forcing)
# Decoder target: [token_ids..., EOS]        ← what decoder should predict

enc_input  = src_ids + [EOS_IDX]
dec_input  = [SOS_IDX] + tgt_ids
dec_target = tgt_ids + [EOS_IDX]

print(f"\n--- After adding special tokens ---")
print(f"Encoder input:  {enc_input}  → {src_tokenizer.decode(src_ids)} <eos>")
print(f"Decoder input:  {dec_input}  → <sos> {tgt_tokenizer.decode(tgt_ids)}")
print(f"Decoder target: {dec_target} → {tgt_tokenizer.decode(tgt_ids)} <eos>")

Vocab size: 28
Vocab size: 24

Source: 'How are you?'
Source IDs: [4, 5, 6]

Target: 'Comment allez-vous?'
Target IDs: [4, 5]

--- After adding special tokens ---
Encoder input:  [4, 5, 6, 2]  → how are you? <eos>
Decoder input:  [1, 4, 5]  → <sos> comment allez-vous?
Decoder target: [4, 5, 2] → comment allez-vous? <eos>


### Key Takeaway

- Tokenization converts raw text → integer IDs using a vocabulary
- Encoder-decoder needs **two separate tokenizers** (source vocab ≠ target vocab)
- **Decoder input is shifted right** — it starts with `<sos>` so the model learns to predict the first real token
- **Decoder target ends with `<eos>`** — so the model learns when to stop generating

**Next Step →** Embedding: converting these integer IDs into dense vectors.

---
## Step 3: Embedding

### What?
A **lookup table** that maps each token ID to a dense vector of fixed size (e.g., 512 dimensions). These vectors are **learned during training** and capture semantic meaning.

### Why?
- Token IDs are just arbitrary integers ("how" = 4, "are" = 5) — they carry **no meaning or relationship**.
- Embeddings place tokens in a continuous vector space where **similar words are close together**.
- The model needs dense, differentiable representations to learn via gradient descent.

### Where in the pipeline?
```
Input Text Corpus → Tokenization → 👉 [Embedding] → Positional Encoding → Encoder → Decoder → Output
```

### Encoder-Decoder specifics
- **Separate embedding layers** for source and target (different vocabularies, different languages)
- Both produce vectors of the same dimension `d_model` (e.g., 512) so they can flow through the same architecture
- Embeddings are scaled by `√(d_model)` as per the original "Attention Is All You Need" paper to stabilize training

In [6]:
# Step 3: Embedding — Token IDs → Dense Vectors

import torch
import torch.nn as nn
import math

class TokenEmbedding(nn.Module):
    """Converts token IDs to scaled dense vectors."""
    
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.d_model = d_model
    
    def forward(self, x):
        # Scale by sqrt(d_model) to prevent embeddings from being too small
        # relative to positional encodings that get added next
        return self.embedding(x) * math.sqrt(self.d_model)

# Config
d_model = 512  # embedding dimension (same as original transformer)
src_vocab_size = len(src_tokenizer.word2idx)
tgt_vocab_size = len(tgt_tokenizer.word2idx)

# Separate embeddings for source and target
src_embedding = TokenEmbedding(src_vocab_size, d_model)
tgt_embedding = TokenEmbedding(tgt_vocab_size, d_model)

# Convert our tokenized example to tensors
enc_input_tensor = torch.tensor([enc_input])   # shape: (1, seq_len)
dec_input_tensor = torch.tensor([dec_input])   # shape: (1, seq_len)

# Get embeddings
# Key outputs for the following process:
# 1. src_embedded: Dense vector representations of source tokens for encoder input
# 2. tgt_embedded: Dense vector representations of target tokens for decoder input
# These embeddings will be combined with positional encodings in the next step
src_embedded = src_embedding(enc_input_tensor)  # (1, src_len, 512)
tgt_embedded = tgt_embedding(dec_input_tensor)  # (1, tgt_len, 512)

print(f"Source vocab size: {src_vocab_size}")
print(f"Target vocab size: {tgt_vocab_size}")
print(f"\nEncoder input tensor: {enc_input_tensor} → shape {enc_input_tensor.shape}")
print(f"Source embedded shape: {src_embedded.shape}  → (batch, seq_len, d_model)")
print(f"\nDecoder input tensor: {dec_input_tensor} → shape {dec_input_tensor.shape}")
print(f"Target embedded shape: {tgt_embedded.shape}  → (batch, seq_len, d_model)")
print(f"\nFirst token embedding (first 10 dims): {src_embedded[0, 0, :10].detach()}")
print(f"First token embedding (first 10 dims): {tgt_embedded[0, 0, :10].detach()}")


Source vocab size: 28
Target vocab size: 24

Encoder input tensor: tensor([[4, 5, 6, 2]]) → shape torch.Size([1, 4])
Source embedded shape: torch.Size([1, 4, 512])  → (batch, seq_len, d_model)

Decoder input tensor: tensor([[1, 4, 5]]) → shape torch.Size([1, 3])
Target embedded shape: torch.Size([1, 3, 512])  → (batch, seq_len, d_model)

First token embedding (first 10 dims): tensor([  2.3115,   7.6997, -21.8821,  12.3027, -20.0213, -18.0008, -47.1393,
         11.1895, -24.0061,  -7.9497])
First token embedding (first 10 dims): tensor([ -7.7574,   5.1499,  16.8749,   6.7017,  -0.4561, -27.5807,  -0.9767,
          4.2387,  26.5902,  14.5366])


### Key Takeaway

- Embedding is a **learnable lookup table**: token ID → dense vector of size `d_model`
- Encoder and decoder have **separate embedding layers** (different vocabularies)
- Scaling by `√(d_model)` keeps embedding magnitudes balanced with positional encodings
- Output shape: `(batch_size, sequence_length, d_model)`

**Next Step →** Positional Encoding: adding position information since transformers have no built-in sense of order.

---
## Step 4: Positional Encoding

### What?
A fixed (non-learned) signal **added** to embeddings that encodes the **position** of each token in the sequence using sine and cosine functions of different frequencies.

### Why?
- Transformers process all tokens **in parallel** (unlike RNNs which go step-by-step).
- Without positional info, "dog bites man" and "man bites dog" would look **identical** to the model.
- Sinusoidal encoding allows the model to generalize to **longer sequences** than seen during training.

### Where in the pipeline?
```
Input Text Corpus → Tokenization → Embedding → 👉 [Positional Encoding] → Encoder → Decoder → Output
```

### The Math
```
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```
- `pos` = position in the sequence (0, 1, 2, ...)
- `i` = dimension index
- Each dimension gets a unique frequency — the model can learn to attend to relative positions

In [7]:
# Step 4: Positional Encoding — Injecting Order Information

class PositionalEncoding(nn.Module):
    """Adds sinusoidal positional encoding to embeddings."""
    
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix: (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()  # (max_len, 1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)  # even dimensions
        pe[:, 1::2] = torch.cos(position * div_term)  # odd dimensions
        
        pe = pe.unsqueeze(0)  # (1, max_len, d_model) — broadcast across batch
        self.register_buffer('pe', pe)  # not a learnable parameter
    
    def forward(self, x):
        # x shape: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]  # add positional encoding
        return self.dropout(x)

# Same positional encoding is shared by both encoder and decoder
pos_encoder = PositionalEncoding(d_model, dropout=0.1)

# Add positional encoding to embeddings
src_pos_encoded = pos_encoder(src_embedded)  # (1, src_len, 512)
tgt_pos_encoded = pos_encoder(tgt_embedded)  # (1, tgt_len, 512)

print(f"Source after positional encoding: {src_pos_encoded.shape}")
print(f"Target after positional encoding: {tgt_pos_encoded.shape}")

# Visualize: embedding vs embedding + positional encoding
print(f"\nBefore PE (first 5 dims): {src_embedded[0, 0, :5].detach()}")
print(f"After  PE (first 5 dims): {src_pos_encoded[0, 0, :5].detach()}")
print(f"\nNotice the values shifted — position info is now baked in!")

print(f"\n--- Summary ---")
print(f"Corpus: {len(corpus)} pairs")
print(f"Source vocab size: {src_vocab_size}")
print(f"Target vocab size: {tgt_vocab_size}")
print(f"Encoder input shape: {src_embedded.shape} → after PE: {src_pos_encoded.shape}")
print(f"Decoder input shape: {tgt_embedded.shape} → after PE: {tgt_pos_encoded.shape}")

print(f"\nBefore PE (first 5 dims): {tgt_embedded[0, 0, :5].detach()}")
print(f"After  PE (first 5 dims): {tgt_pos_encoded[0, 0, :5].detach()}")


Source after positional encoding: torch.Size([1, 4, 512])
Target after positional encoding: torch.Size([1, 3, 512])

Before PE (first 5 dims): tensor([  2.3115,   7.6997, -21.8821,  12.3027, -20.0213])
After  PE (first 5 dims): tensor([  2.5684,   9.6664, -24.3135,  14.7808, -22.2459])

Notice the values shifted — position info is now baked in!

--- Summary ---
Corpus: 8 pairs
Source vocab size: 28
Target vocab size: 24
Encoder input shape: torch.Size([1, 4, 512]) → after PE: torch.Size([1, 4, 512])
Decoder input shape: torch.Size([1, 3, 512]) → after PE: torch.Size([1, 3, 512])

Before PE (first 5 dims): tensor([-7.7574,  5.1499, 16.8749,  6.7017, -0.4561])
After  PE (first 5 dims): tensor([-8.6193,  6.8333, 18.7499,  8.5575, -0.5068])


### Key Takeaway

- Positional encoding is **added** (not concatenated) to embeddings
- Uses **sinusoidal functions** — no extra learnable parameters
- Same PE module is used for both encoder and decoder inputs
- After this step, each token vector contains both **semantic meaning** (from embedding) and **position info** (from PE)

**Next Step →** Encoder Stack: the core of understanding the source sequence.

---
## Step 5: Encoder Stack

### What?
A stack of N identical layers (typically 6), each containing:
1. **Multi-Head Self-Attention** — each source token attends to all other source tokens
2. **Feed-Forward Network (FFN)** — position-wise non-linear transformation
3. **Layer Normalization + Residual Connections** around each sub-layer

### Why?
- Self-attention lets the encoder build **contextualized representations** — the word "bank" gets different representations in "river bank" vs "bank account".
- Stacking layers allows progressively **deeper understanding** of the source.
- The encoder output becomes the **memory** that the decoder cross-attends to.

### Where in the pipeline?
```
Input → Tokenization → Embedding → Positional Encoding → 👉 [Encoder Stack] → Encoder Output → Decoder → Output
```

### Inside one Encoder Layer
```
Input → Multi-Head Self-Attention → Add & Norm → Feed-Forward → Add & Norm → Output
  └─────── residual ────────┘          └──── residual ────┘
```

In [ ]:
# Step 5: Encoder Stack
# Input: Takes positionally-encoded source embeddings (src_pos_encoded) from previous steps
# Output: Generates contextualized representations (encoder_output) that serve as memory for decoder
# Architecture: 6 layers, each containing Multi-Head Attention + Feed-Forward Network
# Flow: Input → Layer1 → Layer2 → ... → Layer6 → Final LayerNorm → Output

class MultiHeadAttention(nn.Module):
    """Multi-Head Attention — the core mechanism of transformers."""
    # Takes query, key, value tensors and computes attention weights
    # Allows model to focus on different positions simultaneously through multiple heads
    
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        # Linear projections and reshape to (B, n_heads, seq_len, d_k)
        Q = self.W_q(query).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = self.dropout(torch.softmax(scores, dim=-1))
        
        # Combine heads
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, -1, self.n_heads * self.d_k)
        return self.W_o(out)


class FeedForward(nn.Module):
    """Position-wise Feed-Forward Network."""
    # Applies non-linear transformations to each position independently
    # Expands to d_ff=2048 dimensions then back to d_model=512
    
    def __init__(self, d_model, d_ff=2048, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        return self.net(x)


class EncoderLayer(nn.Module):
    """Single encoder layer: Self-Attention + FFN with residual & norm."""
    # Each layer has 2 sub-layers: 1) Multi-Head Self-Attention 2) Feed-Forward Network
    # Both use residual connections and layer normalization
    
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, src, src_mask=None):
        # Sub-layer 1: Multi-Head Self-Attention + Residual + Norm
        attn_out = self.self_attn(src, src, src, src_mask)
        src = self.norm1(src + self.dropout1(attn_out))
        # Sub-layer 2: Feed-Forward + Residual + Norm
        ffn_out = self.ffn(src)
        src = self.norm2(src + self.dropout2(ffn_out))
        return src


class Encoder(nn.Module):
    """Stack of N encoder layers."""
    # Contains 6 identical encoder layers stacked sequentially
    # Final layer normalization applied to the output
    
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, src, src_mask=None):
        for layer in self.layers:
            src = layer(src, src_mask)
        return self.norm(src)

# Config
n_layers = 6
n_heads = 8
d_ff = 2048
dropout = 0.1

encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)

# Pass positionally-encoded source through encoder
encoder.eval()
with torch.no_grad():
    encoder_output = encoder(src_pos_encoded)

print(f"Encoder input shape:  {src_pos_encoded.shape}")
print(f"Encoder output shape: {encoder_output.shape}")
print(f"\nEncoder output is the MEMORY that decoder will cross-attend to.")
print(f"Each source token now has a rich, contextualized 512-dim representation.")


Encoder input shape:  torch.Size([1, 4, 512])
Encoder output shape: torch.Size([1, 4, 512])

Encoder output is the MEMORY that decoder will cross-attend to.
Each source token now has a rich, contextualized 512-dim representation.


### Key Takeaway

- Encoder = N stacked layers of (Self-Attention + FFN) with residual connections
- Self-attention lets every source token "see" every other source token
- Output shape is unchanged: `(batch, src_len, d_model)` — but now deeply contextualized
- This encoder output becomes the **memory/context** for the decoder's cross-attention

**Next Step →** Encoder Output: understanding what the encoder produces and how the decoder uses it.

---
## Step 6: Encoder Output (Memory)

### What?
The encoder output is a tensor of shape `(batch, src_len, d_model)` — a **contextualized representation** of every source token. This is often called the **memory** because the decoder "remembers" the source through it.

### Why is this a separate step?
- In encoder-decoder models, the encoder runs **once** per input, but the decoder runs **multiple times** (once per generated token).
- The encoder output is **cached and reused** across all decoder steps — this is a key efficiency optimization.
- During cross-attention, the decoder uses encoder output as **Key** and **Value**, while the decoder's own state is the **Query**.

### Where in the pipeline?
```
Input → Tokenization → Embedding → PE → Encoder Stack → 👉 [Encoder Output] → Decoder Stack → Output
                                                              │
                                                    (cached as memory)
                                                              │
                                                    fed to decoder's
                                                    cross-attention at
                                                    EVERY decode step
```

In [10]:
# Step 6: Encoder Output — The Memory Bridge

# The encoder output is already computed above. Let's inspect it.
memory = encoder_output  # alias used in most codebases

print(f"Memory (encoder output) shape: {memory.shape}")
print(f"  - Batch size:      {memory.shape[0]}")
print(f"  - Source seq len:  {memory.shape[1]} tokens")
print(f"  - d_model:         {memory.shape[2]} dimensions")
print(f"{memory[0, 0, :10].detach()} ... (first token's memory vector)")

print(f"\nHow the decoder uses this memory:")
print(f"  Cross-Attention Key (K)   = Linear(memory)")
print(f"  Cross-Attention Value (V) = Linear(memory)")
print(f"  Cross-Attention Query (Q) = Linear(decoder_state)")
print(f"\n  This lets each decoder token 'look at' all source tokens")
print(f"  to decide what source information is relevant for generating the next target token.")

Memory (encoder output) shape: torch.Size([1, 4, 512])
  - Batch size:      1
  - Source seq len:  4 tokens
  - d_model:         512 dimensions
tensor([ 0.4405,  0.9543, -0.0585,  0.6144, -1.2774,  0.9070, -1.2717,  0.7748,
        -0.4799,  0.8622]) ... (first token's memory vector)

How the decoder uses this memory:
  Cross-Attention Key (K)   = Linear(memory)
  Cross-Attention Value (V) = Linear(memory)
  Cross-Attention Query (Q) = Linear(decoder_state)

  This lets each decoder token 'look at' all source tokens
  to decide what source information is relevant for generating the next target token.


### Key Takeaway

- Encoder output = **memory** = contextualized source representations
- Computed **once**, reused at **every** decoder step (efficiency!)
- Decoder cross-attention: `Q` from decoder, `K` and `V` from encoder output
- This is what makes encoder-decoder different from decoder-only: the decoder has **direct access** to the full source understanding

**Next Step →** Decoder Stack: generating the target sequence using masked self-attention + cross-attention + FFN.

---
## Step 7: Decoder Stack (Masked Self-Attention + Cross-Attention + FFN)

### What?
A stack of N identical layers (typically 6), each containing **three** sub-layers:
1. **Masked Multi-Head Self-Attention** — target tokens attend to previous target tokens only (causal)
2. **Multi-Head Cross-Attention** — target tokens attend to all source tokens (encoder output)
3. **Feed-Forward Network** — position-wise non-linear transformation

### Why three sub-layers?
| Sub-layer | Purpose |
|-----------|---------|
| Masked Self-Attention | Understand what the decoder has generated **so far** (can't peek at future tokens) |
| Cross-Attention | Look at the **source** to decide what to generate next |
| FFN | Refine and transform the combined information |

### Where in the pipeline?
```
Input → ... → Encoder Stack → Encoder Output → 👉 [Decoder Stack] → Output Token Prediction
```

### Inside one Decoder Layer
```
Target Input
    │
    ├─→ Masked Self-Attention ─→ Add & Norm
    │         (Q=K=V from target, future masked)
    │
    ├─→ Cross-Attention ──────→ Add & Norm
    │         (Q from target, K&V from encoder output)
    │
    └─→ Feed-Forward ─────────→ Add & Norm ─→ Output
```

### Why masking?
During training, the decoder sees the **entire** target at once (teacher forcing). The causal mask ensures position `i` can only attend to positions `0..i`, preventing the model from "cheating" by looking at future tokens.

In [11]:
# Step 7: Decoder Stack

class DecoderLayer(nn.Module):
    """Single decoder layer: Masked Self-Attn + Cross-Attn + FFN."""
    
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.masked_self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
    
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        # Sub-layer 1: Masked Self-Attention (target attends to itself, causally)
        attn1 = self.masked_self_attn(tgt, tgt, tgt, tgt_mask)
        tgt = self.norm1(tgt + self.dropout1(attn1))
        
        # Sub-layer 2: Cross-Attention (target attends to encoder output)
        # Q = target state, K & V = encoder output (memory)
        attn2 = self.cross_attn(tgt, memory, memory, memory_mask)
        tgt = self.norm2(tgt + self.dropout2(attn2))
        
        # Sub-layer 3: Feed-Forward
        ffn_out = self.ffn(tgt)
        tgt = self.norm3(tgt + self.dropout3(ffn_out))
        return tgt


class Decoder(nn.Module):
    """Stack of N decoder layers."""
    
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        for layer in self.layers:
            tgt = layer(tgt, memory, tgt_mask, memory_mask)
        return self.norm(tgt)


def generate_causal_mask(size):
    """Creates upper-triangular mask to prevent attending to future tokens."""
    mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
    return ~mask  # True = allowed, False = blocked


# Build decoder
decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

# Create causal mask for target sequence
tgt_seq_len = tgt_pos_encoded.size(1)
tgt_mask = generate_causal_mask(tgt_seq_len).unsqueeze(0).unsqueeze(0)  # (1, 1, tgt_len, tgt_len)

print(f"Causal mask for target (seq_len={tgt_seq_len}):")
print(tgt_mask.squeeze())
print(f"\nTrue = can attend, False = blocked (future tokens)")

# Pass through decoder
decoder.eval()
with torch.no_grad():
    decoder_output = decoder(tgt_pos_encoded, memory, tgt_mask)

print(f"\nDecoder input shape:  {tgt_pos_encoded.shape}")
print(f"Memory shape:         {memory.shape}")
print(f"Decoder output shape: {decoder_output.shape}")

Causal mask for target (seq_len=3):
tensor([[ True, False, False],
        [ True,  True, False],
        [ True,  True,  True]])

True = can attend, False = blocked (future tokens)

Decoder input shape:  torch.Size([1, 3, 512])
Memory shape:         torch.Size([1, 4, 512])
Decoder output shape: torch.Size([1, 3, 512])


### Key Takeaway

- Decoder has **3 sub-layers** per layer (vs encoder's 2): masked self-attn + cross-attn + FFN
- **Causal mask** prevents the decoder from seeing future target tokens during training
- **Cross-attention** is the bridge: decoder queries the encoder's memory to know what source info is relevant
- Output shape: `(batch, tgt_len, d_model)` — one vector per target position

**Next Step →** Output Token Prediction Head: converting decoder vectors into vocabulary probabilities.

---
## Step 8: Output Token Prediction Head

### What?
A **linear projection** layer that maps each decoder output vector (`d_model` dims) to the **target vocabulary size**, followed by softmax to get token probabilities.

### Why?
- The decoder outputs vectors of size `d_model` (e.g., 512) — but we need to pick a **word** from the vocabulary.
- The linear layer projects `d_model → tgt_vocab_size`, giving a score for every possible next token.
- Softmax converts these scores (logits) into a probability distribution.

### Where in the pipeline?
```
... → Encoder → Encoder Output → Decoder Stack → 👉 [Output Token Prediction Head] → Loss
```

### The projection
```
Decoder output: (batch, tgt_len, 512)
     │
     ├─→ Linear(512, vocab_size) → logits: (batch, tgt_len, vocab_size)
     │
     └─→ Softmax → probabilities: (batch, tgt_len, vocab_size)
```

**Note:** During training, we use raw logits with CrossEntropyLoss (it applies softmax internally). Softmax is only explicit during inference.

In [12]:
# Step 8: Output Token Prediction Head

class OutputHead(nn.Module):
    """Projects decoder output to vocabulary logits."""
    
    def __init__(self, d_model, tgt_vocab_size):
        super().__init__()
        self.projection = nn.Linear(d_model, tgt_vocab_size)
    
    def forward(self, decoder_output):
        # decoder_output: (batch, tgt_len, d_model)
        # returns logits:  (batch, tgt_len, tgt_vocab_size)
        return self.projection(decoder_output)


output_head = OutputHead(d_model, tgt_vocab_size)

with torch.no_grad():
    logits = output_head(decoder_output)

print(f"Decoder output shape: {decoder_output.shape}")
print(f"Logits shape:         {logits.shape}  → (batch, tgt_len, vocab_size={tgt_vocab_size})")

# Get predicted tokens
probs = torch.softmax(logits, dim=-1)
predicted_ids = torch.argmax(probs, dim=-1)

print(f"\nPredicted token IDs: {predicted_ids[0].tolist()}")
print(f"Predicted tokens:    {tgt_tokenizer.decode(predicted_ids[0].tolist())}")
print(f"\n(Random predictions since model is untrained — this is expected!)")
print(f"After training, these would match the target: '{corpus[0][1]}'")

Decoder output shape: torch.Size([1, 3, 512])
Logits shape:         torch.Size([1, 3, 24])  → (batch, tgt_len, vocab_size=24)

Predicted token IDs: [15, 6, 15]
Predicted tokens:    appelez-vous? je appelez-vous?

(Random predictions since model is untrained — this is expected!)
After training, these would match the target: 'Comment allez-vous?'


### Key Takeaway

- Output head: `Linear(d_model → vocab_size)` — one score per vocabulary token
- During training: use raw logits with CrossEntropyLoss
- During inference: apply softmax + argmax (or sampling) to get predicted tokens
- Untrained model produces random predictions — training fixes this

**Next Step →** Loss: measuring how wrong the predictions are.

---
## Step 9: Loss

### What?
**Cross-Entropy Loss** — measures the difference between the model's predicted probability distribution and the actual target token at each position.

### Why?
- The model outputs a probability distribution over the entire vocabulary at each position.
- We need a **single number** (the loss) that tells us how wrong the model is.
- Cross-entropy is the standard loss for classification tasks — and next-token prediction is classification over the vocabulary.
- Lower loss = model's predictions are closer to the actual target tokens.

### Where in the pipeline?
```
... → Decoder Stack → Output Token Prediction Head → 👉 [Loss] → Backpropagation
```

### How it works
```
Logits:  [0.1, 0.3, 8.5, 0.2, ...]  ← model's raw scores for each vocab token
Target:  [2]                          ← actual token ID

CrossEntropyLoss = -log(softmax(logits)[target_id])

If model is confident and correct → loss is LOW
If model is wrong or uncertain    → loss is HIGH
```

**Important:** We ignore padding tokens in the loss using `ignore_index=PAD_IDX`.

In [ ]:
# Step 9: Loss — Measuring Prediction Error

# CrossEntropyLoss expects:
#   input:  (batch * tgt_len, vocab_size)  ← logits
#   target: (batch * tgt_len)               ← actual token IDs

# Input: logits tensor from decoder with shape (batch, tgt_len, vocab_size) and target token IDs
# Output: scalar loss value representing prediction error
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)

# Our target (what the decoder should predict)
# Input: dec_target list of token IDs
# Output: tensor with shape (1, tgt_len)
dec_target_tensor = torch.tensor([dec_target])  # (1, tgt_len)

# Reshape for loss computation
# logits: (batch, tgt_len, vocab_size) → (batch*tgt_len, vocab_size)
# target: (batch, tgt_len) → (batch*tgt_len)
# Input: logits tensor with shape (batch, tgt_len, vocab_size)
# Output: flattened logits tensor with shape (batch*tgt_len, vocab_size)
logits_flat = logits.view(-1, tgt_vocab_size)
# Input: target tensor with shape (batch, tgt_len)
# Output: flattened target tensor with shape (batch*tgt_len)
target_flat = dec_target_tensor.view(-1)

# Input: flattened logits and target tensors
# Output: scalar loss value
loss = criterion(logits_flat, target_flat)

print(f"Logits shape (flat):  {logits_flat.shape}  → (tgt_len, vocab_size)")
print(f"Target shape (flat):  {target_flat.shape}   → (tgt_len)")
print(f"Target token IDs:     {target_flat.tolist()}")
print(f"Target tokens:        {tgt_tokenizer.decode(target_flat.tolist())}")
print(f"\nLoss: {loss.item():.4f}")
print(f"\n(High loss is expected for untrained model — random predictions are far from correct)")
print(f"A well-trained model would have loss close to 0.")


Logits shape (flat):  torch.Size([3, 24])  → (tgt_len, vocab_size)
Target shape (flat):  torch.Size([3])   → (tgt_len)
Target token IDs:     [4, 5, 2]
Target tokens:        comment allez-vous? <eos>

Loss: 3.8080

(High loss is expected for untrained model — random predictions are far from correct)
A well-trained model would have loss close to 0.


### Key Takeaway

- **CrossEntropyLoss** = standard loss for token prediction tasks
- `ignore_index=PAD_IDX` ensures padding tokens don't affect the loss
- `label_smoothing=0.1` prevents the model from being overconfident (regularization)
- Loss is computed at **every target position** and averaged

**Next Step →** Backpropagation: using the loss to update model weights.

---
## Step 10: Backpropagation & Optimization

### What?
**Backpropagation** computes gradients of the loss with respect to every learnable parameter. The **optimizer** then updates those parameters to reduce the loss.

### Why?
- The loss tells us *how wrong* the model is, but not *how to fix it*.
- Gradients tell us the **direction** to adjust each parameter to reduce the loss.
- The optimizer (Adam) controls the **step size** and uses momentum for stable training.

### Where in the pipeline?
```
... → Output Token Prediction Head → Loss → 👉 [Backpropagation] → Saved Weights
```

### The training loop
```
for each batch:
    1. Forward pass  → compute logits
    2. Compute loss   → compare logits vs targets
    3. loss.backward() → compute gradients
    4. optimizer.step() → update weights
    5. optimizer.zero_grad() → reset gradients for next batch
```

### Transformer-specific training details
- **Learning rate warmup**: start small, ramp up, then decay (prevents early instability)
- **Adam optimizer** with `β1=0.9, β2=0.98, ε=1e-9` (from original paper)
- **Gradient clipping** to prevent exploding gradients

In [14]:
# Step 10: Backpropagation & Optimization — Full Training Loop

# Assemble the full Encoder-Decoder Transformer
class EncoderDecoderTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, n_heads, n_layers, d_ff, dropout=0.1):
        super().__init__()
        self.src_embedding = TokenEmbedding(src_vocab_size, d_model)
        self.tgt_embedding = TokenEmbedding(tgt_vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, dropout=dropout)
        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.output_head = OutputHead(d_model, tgt_vocab_size)
    
    def encode(self, src, src_mask=None):
        return self.encoder(self.pos_encoding(self.src_embedding(src)), src_mask)
    
    def decode(self, tgt, memory, tgt_mask=None, memory_mask=None):
        return self.decoder(self.pos_encoding(self.tgt_embedding(tgt)), memory, tgt_mask, memory_mask)
    
    def forward(self, src, tgt, tgt_mask=None, src_mask=None):
        memory = self.encode(src, src_mask)
        decoder_out = self.decode(tgt, memory, tgt_mask)
        return self.output_head(decoder_out)


# Prepare dataset: pad all sequences to same length
def prepare_batch(corpus, src_tokenizer, tgt_tokenizer):
    src_batch, tgt_input_batch, tgt_target_batch = [], [], []
    for src_text, tgt_text in corpus:
        src_ids = src_tokenizer.encode(src_text) + [EOS_IDX]
        tgt_ids = tgt_tokenizer.encode(tgt_text)
        src_batch.append(src_ids)
        tgt_input_batch.append([SOS_IDX] + tgt_ids)
        tgt_target_batch.append(tgt_ids + [EOS_IDX])
    
    # Pad to max length in batch
    def pad(seqs):
        max_len = max(len(s) for s in seqs)
        return torch.tensor([s + [PAD_IDX] * (max_len - len(s)) for s in seqs])
    
    return pad(src_batch), pad(tgt_input_batch), pad(tgt_target_batch)


# Build model
model = EncoderDecoderTransformer(
    src_vocab_size=src_vocab_size, tgt_vocab_size=tgt_vocab_size,
    d_model=d_model, n_heads=n_heads, n_layers=n_layers, d_ff=d_ff, dropout=dropout
)

# Optimizer with transformer-specific settings
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)

# Prepare data
src_batch, tgt_input_batch, tgt_target_batch = prepare_batch(corpus, src_tokenizer, tgt_tokenizer)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Source batch: {src_batch.shape}")
print(f"Target input batch: {tgt_input_batch.shape}")
print(f"Target label batch: {tgt_target_batch.shape}")

# Training loop
model.train()
n_epochs = 100

for epoch in range(n_epochs):
    tgt_mask = generate_causal_mask(tgt_input_batch.size(1)).unsqueeze(0).unsqueeze(0)
    
    logits = model(src_batch, tgt_input_batch, tgt_mask)  # forward pass
    loss = criterion(logits.view(-1, tgt_vocab_size), tgt_target_batch.view(-1))  # compute loss
    
    optimizer.zero_grad()  # reset gradients
    loss.backward()        # compute gradients
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
    optimizer.step()       # update weights
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{n_epochs} | Loss: {loss.item():.4f}")

print(f"\nTraining complete! Final loss: {loss.item():.4f}")

Model parameters: 44,179,480
Source batch: torch.Size([8, 6])
Target input batch: torch.Size([8, 6])
Target label batch: torch.Size([8, 6])
Epoch  20/100 | Loss: 0.6485
Epoch  40/100 | Loss: 0.6330
Epoch  60/100 | Loss: 0.6279
Epoch  80/100 | Loss: 0.6258
Epoch 100/100 | Loss: 0.6255

Training complete! Final loss: 0.6255


### Key Takeaway

- Full model assembled: Embedding + PE + Encoder + Decoder + Output Head
- Training loop: forward → loss → backward → clip gradients → optimizer step
- **Gradient clipping** prevents exploding gradients (common in transformers)
- Loss should decrease over epochs — the model is learning the source→target mapping

**Next Step →** Saved Weights: persisting the trained model.

---
## Step 11: Saved Weights (.pth / .bin / .safetensors)

### What?
Saving the trained model's **weights** (parameters), **config**, and **tokenizer** to disk so the model can be loaded later for inference without retraining.

### Why?
- Training is expensive (hours/days/weeks) — you don't want to retrain every time.
- Saved weights can be shared, deployed, or fine-tuned.
- A complete model = **Weights + Config + Tokenizer**.

### Where in the pipeline?
```
... → Loss → Backpropagation → 👉 [Saved Weights] (.pth/.bin/.safetensors)
```

### File formats
| Format | Used by | Notes |
|--------|---------|-------|
| `.pth` | PyTorch | Standard PyTorch checkpoint |
| `.bin` | HuggingFace | PyTorch weights in HF format |
| `.safetensors` | HuggingFace | Safer, faster loading (recommended for production) |

In [15]:
# Step 11: Saving the Trained Model
import json, os

save_dir = 'trained_enc_dec_model'
os.makedirs(save_dir, exist_ok=True)

# 1. Save model weights
torch.save(model.state_dict(), f'{save_dir}/model.pth')

# 2. Save config (needed to reconstruct the architecture)
config = {
    'src_vocab_size': src_vocab_size, 'tgt_vocab_size': tgt_vocab_size,
    'd_model': d_model, 'n_heads': n_heads, 'n_layers': n_layers,
    'd_ff': d_ff, 'dropout': dropout
}
with open(f'{save_dir}/config.json', 'w') as f:
    json.dump(config, f, indent=2)

# 3. Save tokenizer vocabularies
with open(f'{save_dir}/src_vocab.json', 'w') as f:
    json.dump(src_tokenizer.word2idx, f, indent=2)
with open(f'{save_dir}/tgt_vocab.json', 'w') as f:
    json.dump(tgt_tokenizer.word2idx, f, indent=2)

print(f"Model saved to '{save_dir}/':")
for f in os.listdir(save_dir):
    size = os.path.getsize(f'{save_dir}/{f}')
    print(f"  {f:20s} ({size:,} bytes)")

Model saved to 'trained_enc_dec_model/':
  config.json          (143 bytes)
  model.pth            (187,036,575 bytes)
  src_vocab.json       (431 bytes)
  tgt_vocab.json       (407 bytes)


### Key Takeaway

- A saved model = **Weights + Config + Tokenizer** — all three are needed to reload
- `state_dict()` saves only the learned parameters (not the code)
- Config stores architecture hyperparameters to reconstruct the model
- This completes the **Training Pipeline**!

---

# 🚀 Inference Pipeline

`User Input → Tokenization → Embedding → Positional Encoding → Encoder Stack → Encoder Output → Decoder Stack → Token-by-Token Generation`

**Next Step →** Inference: loading the model and generating translations token by token.

---
## Step 12: Inference — Token-by-Token Generation

### What?
Load the saved model and generate target tokens **one at a time** in an autoregressive loop:
1. Encode the source **once** → get memory
2. Start decoder with `<sos>`
3. Predict next token → append to decoder input → repeat until `<eos>` or max length

### Why is inference different from training?
| | Training | Inference |
|--|---------|-----------|
| Decoder input | Full target (teacher forcing) | Generated so far (autoregressive) |
| Encoder runs | Once per batch | Once per input |
| Decoder runs | Once (parallel) | N times (one per token) |
| Mask | Causal mask on full target | Grows with each step |

### Where in the pipeline?
```
User Input → Tokenize → Embed → PE → Encoder → Memory
                                                  │
                                    ┌─────────────┘
                                    │
              <sos> → Decoder → predict token_1
     <sos> token_1 → Decoder → predict token_2
<sos> token_1 token_2 → Decoder → predict token_3
                  ... until <eos>
```

In [16]:
# Step 12: Inference — Load Model & Generate Token-by-Token

# --- Load saved model ---
with open(f'{save_dir}/config.json') as f:
    cfg = json.load(f)

# Rebuild tokenizers from saved vocab
def load_tokenizer(vocab_path):
    tok = Tokenizer()
    with open(vocab_path) as f:
        tok.word2idx = json.load(f)
    tok.idx2word = {int(v): k for k, v in tok.word2idx.items()}
    return tok

inf_src_tok = load_tokenizer(f'{save_dir}/src_vocab.json')
inf_tgt_tok = load_tokenizer(f'{save_dir}/tgt_vocab.json')

# Rebuild model from config and load weights
inf_model = EncoderDecoderTransformer(**cfg)
inf_model.load_state_dict(torch.load(f'{save_dir}/model.pth', weights_only=True))
inf_model.eval()
print(f"Model loaded from '{save_dir}/'")

Model loaded from 'trained_enc_dec_model/'


In [17]:
# --- Greedy Decoding: Token-by-Token Generation ---

@torch.no_grad()
def translate(model, src_text, src_tokenizer, tgt_tokenizer, max_len=50):
    """Translate source text using greedy decoding."""
    model.eval()
    
    # 1. Tokenize source and encode ONCE
    src_ids = src_tokenizer.encode(src_text) + [EOS_IDX]
    src_tensor = torch.tensor([src_ids])
    memory = model.encode(src_tensor)
    
    # 2. Start decoder with <sos>
    generated = [SOS_IDX]
    
    for step in range(max_len):
        tgt_tensor = torch.tensor([generated])
        tgt_mask = generate_causal_mask(len(generated)).unsqueeze(0).unsqueeze(0)
        
        # 3. Decode and predict next token
        decoder_out = model.decode(tgt_tensor, memory, tgt_mask)
        logits = model.output_head(decoder_out[:, -1, :])  # last position only
        next_token = logits.argmax(dim=-1).item()
        
        # 4. Stop if <eos>
        if next_token == EOS_IDX:
            break
        
        generated.append(next_token)
    
    # Remove <sos> and decode
    return tgt_tokenizer.decode(generated[1:])


# Test on training examples
print("=== Translation Results ===\n")
for src_text, expected in corpus:
    prediction = translate(inf_model, src_text, inf_src_tok, inf_tgt_tok)
    match = '✅' if prediction.strip() == expected.lower() else '❌'
    print(f"{match} '{src_text}'")
    print(f"   Expected:  {expected.lower()}")
    print(f"   Got:       {prediction}\n")

=== Translation Results ===

✅ 'How are you?'
   Expected:  comment allez-vous?
   Got:       comment allez-vous?

✅ 'I am fine.'
   Expected:  je vais bien.
   Got:       je vais bien.

✅ 'Thank you very much.'
   Expected:  merci beaucoup.
   Got:       merci beaucoup.

✅ 'Good morning.'
   Expected:  bonjour.
   Got:       bonjour.

✅ 'See you tomorrow.'
   Expected:  à demain.
   Got:       à demain.

✅ 'What is your name?'
   Expected:  comment vous appelez-vous?
   Got:       comment vous appelez-vous?

✅ 'I love programming.'
   Expected:  j'adore la programmation.
   Got:       j'adore la programmation.

✅ 'The weather is nice today.'
   Expected:  le temps est beau aujourd'hui.
   Got:       le temps est beau aujourd'hui.



### Key Takeaway

- Inference is **autoregressive**: generate one token at a time, feed it back
- Encoder runs **once**, decoder runs **N times** (once per output token)
- **Greedy decoding**: always pick the highest-probability token (simple but not always optimal)
- Production systems use **beam search** or **sampling** (top-k, top-p) for better quality

---

## 🎯 Summary: Complete Encoder-Decoder Pipeline

### Training Pipeline
```
Input Text Corpus          → Parallel source-target pairs
  → Tokenization            → Text → token IDs (separate vocabs)
  → Embedding               → Token IDs → dense vectors (d_model)
  → Positional Encoding     → Add position info (sinusoidal)
  → Encoder Stack           → Self-attention + FFN (N layers)
  → Encoder Output          → Contextualized source memory
  → Decoder Stack           → Masked self-attn + cross-attn + FFN
  → Output Head             → Linear projection → vocab logits
  → Loss                    → CrossEntropy(predicted vs actual)
  → Backpropagation         → Gradients → optimizer → update weights
  → Saved Weights           → .pth + config.json + vocab files
```

### Inference Pipeline
```
User Input                  → Source text
  → Tokenization            → Text → token IDs
  → Embedding + PE          → Dense vectors with position
  → Encoder (once)          → Memory
  → Decoder (loop)          → Token-by-token generation
  → Stop at <eos>           → Final output
```